<a href="https://colab.research.google.com/github/abhisheksvcs25-bms/1BM25CS406_AI_LAB/blob/YT-Hand-Gesture-Controller/1BM25CS406_Week7_FOL_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

# Parse a predicate into its name and arguments
def parse_literal(literal):
    match_obj = re.match(r"^(\w+)\((.*?)\)$", literal.strip())

    if match_obj:
        predicate = match_obj.group(1)
        arguments = match_obj.group(2).split(",")
        arguments = [arg.strip() for arg in arguments]
        return predicate, arguments

    return literal.strip(), None


# Check whether an argument is a variable
# MODIFIED: Variables are lowercase (p, q, r, x) so constants (Robert, America, A, T1) can be uppercase
def is_variable(term):
    return term[0].islower()


# Unification function
def unify(pattern, fact, substitution):
    p_pred, p_args = parse_literal(pattern)
    f_pred, f_args = parse_literal(fact)

    # Predicate names must match
    if p_pred != f_pred:
        return None

    # Handle predicates without arguments
    if p_args is None or f_args is None:
        if pattern == fact:
            return substitution.copy()
        return None

    # Number of arguments must match
    if len(p_args) != len(f_args):
        return None

    new_substitution = substitution.copy()

    for p_arg, f_arg in zip(p_args, f_args):
        # Variable matching
        if is_variable(p_arg):
            if p_arg in new_substitution:
                if new_substitution[p_arg] != f_arg:
                    return None
            else:
                new_substitution[p_arg] = f_arg

        # Constant matching
        elif p_arg != f_arg:
            return None

    return new_substitution


# Find all substitutions satisfying all premises
def find_substitutions(premises, facts, index=0, substitution=None):
    if substitution is None:
        substitution = {}

    # All premises successfully matched
    if index == len(premises):
        return [substitution]

    results = []
    current_premise = premises[index]

    for fact in facts:
        new_substitution = unify(
            current_premise,
            fact,
            substitution
        )

        if new_substitution is not None:
            results.extend(
                find_substitutions(
                    premises,
                    facts,
                    index + 1,
                    new_substitution
                )
            )

    return results


# Substitute variables in a conclusion
def substitute(expression, substitution):
    predicate, arguments = parse_literal(expression)

    if arguments is None:
        return expression

    new_arguments = []
    for arg in arguments:
        new_arguments.append(
            substitution.get(arg, arg)
        )

    return predicate + "(" + ",".join(new_arguments) + ")"


# FOL Forward Chaining Algorithm with verbose logging output
def fol_forward_chaining(KB, query):
    print("Initial Knowledge Base Facts:")
    for fact in sorted(KB["facts"]):
        print(f" - {fact}")
    print("\n" + "="*50)
    print("STARTING FORWARD CHAINING INFERENCES")
    print("="*50)

    # Check whether query already exists
    for fact in KB["facts"]:
        if unify(query, fact, {}) is not None:
            print(f"Query '{query}' found immediately in initial facts!")
            return True

    iteration = 0
    # Repeat until no new facts are generated
    while True:
        iteration += 1
        print(f"\n--- ITERATION {iteration} ---")
        new_facts = set()
        inferred_in_this_iteration = False

        # Check every rule in the Knowledge Base
        for rule in KB["rules"]:
            premises = rule[0]
            conclusion = rule[1]

            # Find substitutions satisfying all premises
            substitutions = find_substitutions(premises, KB["facts"])

            for substitution in substitutions:
                # Infer a new fact
                inferred = substitute(conclusion, substitution)

                # Store only facts not already in KB
                if inferred not in KB["facts"] and inferred not in new_facts:
                    rule_str = " ∧ ".join(premises) + " ⇒ " + conclusion
                    print(f"👉 Rule Matched: {rule_str}")
                    print(f"   Substitution: {substitution}")
                    print(f"   ✨ Inferred Fact: {inferred}\n")
                    new_facts.add(inferred)
                    inferred_in_this_iteration = True

                # Check whether the inferred fact proves query
                if unify(query, inferred, {}) is not None:
                    print(f"🎯 Target Query '{query}' successfully derived!")
                    KB["facts"].update(new_facts)
                    return True

        if not inferred_in_this_iteration:
            print("No new facts could be inferred in this iteration.")

        # Stop if no new facts can be inferred
        if not new_facts:
            print("="*50)
            return False

        # Add newly inferred facts to Knowledge Base
        KB["facts"].update(new_facts)


# Function to display the result
def execute_test(KB, query):
    result = fol_forward_chaining(KB, query)
    print("="*50)
    if result:
        print("Query proved successfully!")
        print("Result: TRUE")
    else:
        print("Query could not be proved.")
        print("Result: FALSE")
    print("="*50)


# -------------------------------------------------
# THE LAW & CRIMINAL CUSTOM TEST CASE
# -------------------------------------------------

KB_CRIME = {
    "facts": {
        "American(Robert)",
        "Enemy(A,America)",
        "Owns(A,T1)",
        "Missile(T1)"
    },

    "rules": [
        # Missile(x) => Weapon(x)
        (
            ["Missile(x)"],
            "Weapon(x)"
        ),
        # Enemy(x, America) => Hostile(x)
        (
            ["Enemy(x,America)"],
            "Hostile(x)"
        ),
        # Missile(x) ∧ Owns(A, x) => Sells(Robert, x, A)
        (
            ["Missile(x)", "Owns(A,x)"],
            "Sells(Robert,x,A)"
        ),
        # American(p) ∧ Weapon(q) ∧ Sells(p, q, r) ∧ Hostile(r) => Criminal(p)
        (
            ["American(p)", "Weapon(q)", "Sells(p,q,r)", "Hostile(r)"],
            "Criminal(p)"
        )
    ]
}

query_crime = "Criminal(Robert)"

execute_test(KB_CRIME, query_crime)


Initial Knowledge Base Facts:
 - American(Robert)
 - Enemy(A,America)
 - Missile(T1)
 - Owns(A,T1)

STARTING FORWARD CHAINING INFERENCES

--- ITERATION 1 ---
👉 Rule Matched: Missile(x) ⇒ Weapon(x)
   Substitution: {'x': 'T1'}
   ✨ Inferred Fact: Weapon(T1)

👉 Rule Matched: Enemy(x,America) ⇒ Hostile(x)
   Substitution: {'x': 'A'}
   ✨ Inferred Fact: Hostile(A)

👉 Rule Matched: Missile(x) ∧ Owns(A,x) ⇒ Sells(Robert,x,A)
   Substitution: {'x': 'T1'}
   ✨ Inferred Fact: Sells(Robert,T1,A)


--- ITERATION 2 ---
👉 Rule Matched: American(p) ∧ Weapon(q) ∧ Sells(p,q,r) ∧ Hostile(r) ⇒ Criminal(p)
   Substitution: {'p': 'Robert', 'q': 'T1', 'r': 'A'}
   ✨ Inferred Fact: Criminal(Robert)

🎯 Target Query 'Criminal(Robert)' successfully derived!
Query proved successfully!
Result: TRUE
